# CellularComponent → ChemicalEntity Relation Pipeline

Builds a unified, deduplicated edge table for the **CellularComponent–ChemicalEntity** relation.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- PrimeKG
- Pheknowlator

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [10]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CELLULARCOMPONENT_CHEMICALENTITY/ALL_CELLULARCOMPONENT_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1 · Mapping DBs

In [11]:
# Pubchem = pd.read_parquet(MAPPING_DIR + 'pubchem/combined_df.paraquet')
# Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
# del Pubchem

## 2 · Load Source Files

In [12]:
# PrimeKG
PrimeKG = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_CellComp_Chemical.csv')
PrimeKG.columns = PrimeKG.columns.str.lower()
PrimeKG['head_id_is'] = 'Quick_GO'
PrimeKG['tail_id_is'] = 'PubChem'
PrimeKG['relation'] = 'CellularComponent_ChemicalEntity'
PrimeKG['head_type'] = 'CellularComponent'
PrimeKG['tail_type'] = 'ChemicalEntity'
PrimeKG['kg_type'] = 'Generalised'
PrimeKG['species'] = 'Homo sapiens'

PrimeKG["tail"] = PrimeKG["tail"].astype(str).str.replace(r'\.0$', '', regex=True)
print(f"PrimeKG: {len(PrimeKG):,} rows")
PrimeKG

PrimeKG: 5 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species
0,GO:0071736,CellularComponent_ChemicalEntity,3781988,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,9-heptadecaflu...",hexadecafluoro-nonanoic acid,C(=O)(C(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F...,C(=O)(C(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F...,Generalised,Homo sapiens
1,GO:0071736,CellularComponent_ChemicalEntity,23685740,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,1...",perfluorodecanoic acid,C(=O)(C(C(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)...,C(=O)(C(C(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)...,Generalised,Homo sapiens
2,GO:0071736,CellularComponent_ChemicalEntity,23704962,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro...",perfluorohexanesulfonic acid,C(C(C(C(F)(F)S(=O)(=O)[O-])(F)F)(F)F)(C(C(F)(F...,C(C(C(C(F)(F)S(=O)(=O)[O-])(F)F)(F)F)(C(C(F)(F...,Generalised,Homo sapiens
3,GO:0071736,CellularComponent_ChemicalEntity,23677927,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","lithium;1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-hept...",perfluorooctane sulfonic acid,[Li+].C(C(C(C(C(F)(F)S(=O)(=O)[O-])(F)F)(F)F)(...,[Li+].C(C(C(C(C(F)(F)S(=O)(=O)[O-])(F)F)(F)F)(...,Generalised,Homo sapiens
4,GO:0071736,CellularComponent_ChemicalEntity,23667657,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-pentadeca...",perfluorooctanoic acid,C(=O)(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F...,C(=O)(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F...,Generalised,Homo sapiens


## 3 · Consolidate

In [13]:
source_dfs = [PrimeKG]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['tail'].astype(str).str.upper() != 'NAN']

In [14]:
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0071736,CellularComponent_ChemicalEntity,3781988,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,9-heptadecaflu...",Generalised,Homo sapiens
1,GO:0071736,CellularComponent_ChemicalEntity,23685740,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,1...",Generalised,Homo sapiens
2,GO:0071736,CellularComponent_ChemicalEntity,23704962,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro...",Generalised,Homo sapiens
3,GO:0071736,CellularComponent_ChemicalEntity,23677927,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","lithium;1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-hept...",Generalised,Homo sapiens
4,GO:0071736,CellularComponent_ChemicalEntity,23667657,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-pentadeca...",Generalised,Homo sapiens


In [15]:
final_df[final_df['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 4 · Chemical Name Normalisation

In [16]:
# tail_ids = pd.to_numeric(final_df['tail'], errors='coerce')
# mask = final_df['tail_detail_name'].isna()
# final_df.loc[mask, 'tail_detail_name'] = tail_ids[mask].astype('str').map(Pubchem_IUPAC_CID_Dict)
final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)
print(f"Consolidated rows: {len(final_df):,}")

Consolidated rows: 5


In [17]:
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0071736,CellularComponent_ChemicalEntity,3781988,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,9-heptadecaflu...",Generalised,Homo sapiens
1,GO:0071736,CellularComponent_ChemicalEntity,23685740,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,1...",Generalised,Homo sapiens
2,GO:0071736,CellularComponent_ChemicalEntity,23704962,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro...",Generalised,Homo sapiens
3,GO:0071736,CellularComponent_ChemicalEntity,23677927,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","lithium;1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-hept...",Generalised,Homo sapiens
4,GO:0071736,CellularComponent_ChemicalEntity,23667657,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-pentadeca...",Generalised,Homo sapiens


In [18]:
# Count true NaN values
true_nan_count = final_df.isna().sum()

# Count string 'NAN' values (case-insensitive)
string_nan_count = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

# Combine both counts
nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})
nan_summary

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 6. Adding column kg_type

In [19]:

final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0071736,CellularComponent_ChemicalEntity,3781988,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,9-heptadecaflu...",Generalised,Homo sapiens
1,GO:0071736,CellularComponent_ChemicalEntity,23685740,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,1...",Generalised,Homo sapiens
2,GO:0071736,CellularComponent_ChemicalEntity,23704962,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro...",Generalised,Homo sapiens
3,GO:0071736,CellularComponent_ChemicalEntity,23677927,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","lithium;1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-hept...",Generalised,Homo sapiens
4,GO:0071736,CellularComponent_ChemicalEntity,23667657,CellularComponent,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,"IgG immunoglobulin complex, circulating","sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-pentadeca...",Generalised,Homo sapiens


## 7 · Save Output

In [20]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df):,} rows → {OUT_PATH}")

Saved 5 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CELLULARCOMPONENT_CHEMICALENTITY/ALL_CELLULARCOMPONENT_CHEMICALENTITY.csv
